# MLOps Pipeline - Exploration Notebook

This notebook demonstrates how to use the MLOps pipeline components for:
- Data ingestion and preprocessing
- Model creation and training
- Experiment tracking with W&B
- Model evaluation


In [ ]:
# Add src to path
import sys
sys.path.insert(0, '../src')

# Standard imports
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# MLOps imports
from src.utils.config import load_config
from src.utils.logging import setup_logging, get_logger
from src.data.preprocessing import Preprocessor, DataSplitter
from src.data.validation import DataValidator
from src.models.architectures import create_model
from src.training.trainer import Trainer
from src.evaluation.metrics import create_classification_metrics

# Setup logging
setup_logging(level='INFO')
logger = get_logger(__name__)

## 1. Load Configuration

In [ ]:
# Load configurations
training_config = load_config('../config/training.yaml')
data_config = load_config('../config/data.yaml')
experiment_config = load_config('../config/experiment.yaml')

print("Training Config:")
print(f"  - Model: {training_config.model.architecture}")
print(f"  - Epochs: {training_config.training.epochs}")
print(f"  - Batch Size: {training_config.training.batch_size}")
print(f"  - Learning Rate: {training_config.training.optimizer.learning_rate}")

## 2. Create Sample Data

In [ ]:
# Generate sample classification data
np.random.seed(42)

n_samples = 1000
n_features = 20
n_classes = 10

X = np.random.randn(n_samples, n_features)
y = np.random.randint(0, n_classes, n_samples)

# Create DataFrame
feature_names = [f'feature_{i}' for i in range(n_features)]
df = pd.DataFrame(X, columns=feature_names)
df['label'] = y

print(f"Dataset shape: {df.shape}")
print(f"\nLabel distribution:")
print(df['label'].value_counts().sort_index())

## 3. Data Validation

In [ ]:
# Validate data
validator = DataValidator(config=data_config)
validation_results = validator.validate(df)

print("Validation Results:")
for result in validation_results:
    print(f"  - {result.check_name}: {result.status.value}")
    if result.message:
        print(f"    Message: {result.message}")

## 4. Data Preprocessing

In [ ]:
# Preprocess data
preprocessor = Preprocessor(config=data_config)
processed_df = preprocessor.fit_transform(df)

print(f"Processed shape: {processed_df.shape}")
print(f"\nProcessed statistics:")
print(processed_df.describe())

In [ ]:
# Split data
splitter = DataSplitter(
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    random_state=42
)

train_df, val_df, test_df = splitter.split(processed_df)

print(f"Train: {len(train_df)} samples")
print(f"Val: {len(val_df)} samples")
print(f"Test: {len(test_df)} samples")

## 5. Create Model

In [ ]:
# Modify config for custom MLP (since we have tabular data)
from omegaconf import OmegaConf

model_config = OmegaConf.create({
    'architecture': 'custom',
    'num_classes': n_classes,
    'input_dim': n_features,
    'dropout_rate': 0.2,
    'custom': {
        'hidden_dims': [128, 64, 32],
        'activation': 'relu',
        'batch_norm': True
    }
})

model = create_model(model_config)

print(f"Model: {model.__class__.__name__}")
print(f"Total parameters: {model.num_parameters:,}")
print(f"Trainable parameters: {model.num_trainable_parameters:,}")

## 6. Create DataLoaders

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

# Prepare tensors
def df_to_tensors(df, label_col='label'):
    X = torch.FloatTensor(df.drop(columns=[label_col]).values)
    y = torch.LongTensor(df[label_col].values)
    return TensorDataset(X, y)

train_dataset = df_to_tensors(train_df)
val_dataset = df_to_tensors(val_df)
test_dataset = df_to_tensors(test_df)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## 7. Train Model

In [ ]:
# Create training config
train_config = OmegaConf.create({
    'model': model_config,
    'training': {
        'epochs': 10,
        'batch_size': batch_size,
        'gradient_accumulation_steps': 1,
        'optimizer': {
            'type': 'adam',
            'learning_rate': 0.001,
            'weight_decay': 0.01
        },
        'scheduler': {
            'type': 'cosine',
            'min_lr': 1e-6
        },
        'loss': {
            'type': 'cross_entropy'
        },
        'mixed_precision': {
            'enabled': False
        },
        'gradient_clipping': {
            'enabled': True,
            'max_norm': 1.0
        },
        'early_stopping': {
            'enabled': True,
            'patience': 5,
            'monitor': 'val_loss',
            'mode': 'min'
        }
    },
    'checkpointing': {
        'enabled': False
    },
    'distributed': {
        'enabled': False
    },
    'wandb': {
        'mode': 'disabled'  # Disable for notebook
    }
})

# Create trainer
trainer = Trainer(
    model=model,
    config=train_config,
    train_loader=train_loader,
    val_loader=val_loader
)

# Train
results = trainer.train()

print(f"\nTraining completed!")
print(f"Best epoch: {results['best_epoch']}")
print(f"Best metric: {results['best_metric']:.4f}")

## 8. Evaluate Model

In [ ]:
from src.evaluation.evaluator import Evaluator

# Evaluate on test set
evaluator = Evaluator(
    model=model,
    data_loader=test_loader,
    task_type='classification'
)

eval_results = evaluator.evaluate()

print("Evaluation Results:")
for metric, value in eval_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.4f}")

In [ ]:
# Display confusion matrix
if 'confusion_matrix' in eval_results:
    cm = eval_results['confusion_matrix']
    
    plt.figure(figsize=(10, 8))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('Confusion Matrix')
    plt.colorbar()
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.show()

## 9. Save Model

In [ ]:
# Save model checkpoint
import os

checkpoint_dir = '../checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

checkpoint_path = os.path.join(checkpoint_dir, 'notebook_model.pt')
model.save(checkpoint_path)

print(f"Model saved to: {checkpoint_path}")

## 10. Summary

This notebook demonstrated:
1. Loading YAML configurations
2. Data validation and preprocessing
3. Creating a custom MLP model
4. Training with the Trainer class
5. Evaluating with comprehensive metrics
6. Saving model checkpoints

For production use:
- Enable W&B tracking by setting `wandb.mode: 'online'`
- Use the Airflow DAGs for orchestration
- Register models with the Model Registry
